# Genetic Algorithm
Genetic representation: <br>
List of clients in order of being calculated.

Mutation: <br>
Swap mutation to change the path randomly. Each mutation will use one parent to make sure no duplicate clients get added.

Tournament selection: <br>
Used to get a fair spread of good and bad states.

Elitism survivor selection: <br>
To make sure we do not go back in progress.

In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import osmnx as ox
import numpy as np

In [52]:
# Initialize all necessary parts of the genetic algorithm
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

POPULATION_SIZE = 100
EPOCHS = 500

MUTATION_RATE = 1.00
MUTATION_STRIDE = 2

TOURNAMENT_SIZE = 5
ELITE_SIZE = max(1, int(0.10 * POPULATION_SIZE))

MAX_EMPLOYEE_WORK_TIME_MIN = 8 * 60

print("GA initialized:")
print(f"  seed={RANDOM_SEED}, population={POPULATION_SIZE}, epochs={EPOCHS}")
print(f"  mutation_rate={MUTATION_RATE}, mutation_stride={MUTATION_STRIDE}")
print(f"  tournament={TOURNAMENT_SIZE}, elite_size={ELITE_SIZE}")

GA initialized:
  seed=42, population=100, epochs=500
  mutation_rate=1.0, mutation_stride=2
  tournament=5, elite_size=10


In [46]:
# ============= Initial Population =============
NUM_CLIENTS = 100
client_indices = list(range(NUM_CLIENTS))
initial_population = [np.random.permutation(client_indices).tolist() for _ in range(POPULATION_SIZE)]

print(f"Initialized population with {len(initial_population)} individuals.")
print(f"Each individual contains {len(initial_population[0])} client indices.")
print("First individual:")
print(initial_population[0])

Initialized population with 100 individuals.
Each individual contains 100 client indices.
First individual:
[83, 53, 70, 45, 44, 39, 22, 80, 10, 0, 18, 30, 73, 33, 90, 4, 76, 77, 12, 31, 55, 88, 26, 42, 69, 15, 40, 96, 9, 72, 11, 47, 85, 28, 93, 5, 66, 65, 35, 16, 49, 34, 7, 95, 27, 19, 81, 25, 62, 13, 24, 3, 17, 38, 8, 78, 6, 64, 36, 89, 56, 99, 54, 43, 50, 67, 46, 68, 61, 97, 79, 41, 58, 48, 98, 57, 75, 32, 94, 59, 63, 84, 37, 29, 1, 52, 21, 2, 23, 87, 91, 74, 86, 82, 20, 60, 71, 14, 92, 51]


In [47]:
# ============= Mutation Function =============
def swap_mutation(parent):
    """
    Swap mutation: takes a single parent (list of client indices),
    creates a mutant by swapping MUTATION_STRIDE random position pairs.
    
    Args:
        parent: list of client indices representing a route order
        
    Returns:
        mutant: mutated copy of parent with swapped positions
    """
    mutant = parent.copy()
    
    # Perform MUTATION_STRIDE swaps
    for _ in range(MUTATION_STRIDE):
        # Select two random distinct indices
        i, j = np.random.choice(len(mutant), size=2, replace=False)
        # Swap them
        mutant[i], mutant[j] = mutant[j], mutant[i]
    
    return mutant

In [48]:
# ============= Tournament Selection =============
def tournament_selection(population, fitness_scores):
    """
    Tournament selection: randomly sample TOURNAMENT_SIZE individuals
    and return the best one (lower fitness is better for minimization).
    
    Args:
        population: list of individuals (each individual is a list of client indices)
        fitness_scores: list of fitness values corresponding to each individual
        
    Returns:
        winner: selected individual from the tournament
        winner_fitness: fitness score of the selected individual
    """
    # Sample unique indices for the tournament
    tournament_size = min(TOURNAMENT_SIZE, len(population))
    candidate_indices = np.random.choice(len(population), size=tournament_size, replace=False)
    
    # Find index of the best candidate in the sampled set (lower fitness is better)
    best_local_idx = np.argmin([fitness_scores[i] for i in candidate_indices])
    winner_index = candidate_indices[best_local_idx]
    
    winner = population[winner_index]
    winner_fitness = fitness_scores[winner_index]
    
    return winner, winner_fitness

In [49]:
# ============= Elitism Selection =============
def elitism_selection(population, fitness_scores):
    """
    Elitism selection: preserve the top ELITE_SIZE individuals from the population
    based on their fitness scores (lower fitness is better for minimization).
    
    Args:
        population: list of individuals (each individual is a list of client indices)
        fitness_scores: list of fitness values corresponding to each individual
        
    Returns:
        elite_individuals: list of the ELITE_SIZE best individuals
        elite_fitness: list of fitness scores for the elite individuals
    """
    # Get indices sorted by fitness (ascending - lower is better)
    sorted_indices = np.argsort(fitness_scores)[:ELITE_SIZE]
    
    # Extract elite individuals and their fitness scores
    elite_individuals = [population[i] for i in sorted_indices]
    elite_fitness = [fitness_scores[i] for i in sorted_indices]
    
    return elite_individuals, elite_fitness

In [ ]:
clients_with_coords = pd.read_csv("../output/clients_with_coords.csv")
heerlen_edge_table = pd.read_csv("../output/heerlen_edge_table.csv")



for i in range(len(initial_population[0])):
    client_index = initial_population[0][i]
    client_info = clients_with_coords.iloc[client_index]
    coords = (client_info['longitude'], client_info['latitude'])
    #print(coords)

for i in range(len(heerlen_edge_table)):
    edge_info = heerlen_edge_table.iloc[i]
    geometry = str(edge_info['geometry'])
    if geometry.upper().startswith('LINESTRING'):
        start = geometry.find('(')
        end = geometry.rfind(')')
        geometry = geometry[start + 1:end] if start != -1 and end != -1 else geometry
    #print(geometry)




(np.float64(5.98154), np.float64(50.88365))
(np.float64(5.9818004), np.float64(50.8905669))
(np.float64(5.98154), np.float64(50.88365))
(np.float64(5.98154), np.float64(50.88365))
(np.float64(5.9926908), np.float64(50.8645727))
(np.float64(5.98154), np.float64(50.88365))
(np.float64(5.9996203), np.float64(50.8856488))
(np.float64(5.98154), np.float64(50.88365))
(np.float64(5.98154), np.float64(50.88365))
(np.float64(5.98154), np.float64(50.88365))
(np.float64(5.98154), np.float64(50.88365))
(np.float64(5.9798024), np.float64(50.8827826))
(np.float64(5.98154), np.float64(50.88365))
(np.float64(5.98154), np.float64(50.88365))
(np.float64(5.98154), np.float64(50.88365))
(np.float64(5.98154), np.float64(50.88365))
(np.float64(5.98154), np.float64(50.88365))
(np.float64(5.98154), np.float64(50.88365))
(np.float64(5.98154), np.float64(50.88365))
(np.float64(5.98154), np.float64(50.88365))
(np.float64(5.98154), np.float64(50.88365))
(np.float64(5.98154), np.float64(50.88365))
(np.float64(5.98